# FinIntent DataPrep — Create Labeled Intent Dataset

## Goal
Create a clean, balanced, labeled dataset for intent classification.

## Steps
1. Load Banking77 (13,083 queries, 77 intents)
2. Map 77 intents → 4 PDF categories, drop irrelevant rows
3. Check class balance → Regulatory will be underrepresented
4. Generate 500 Regulatory questions from Regulation E.pdf using Groq
5. Final balanced dataset → visualize → save as CSV

## 4 Intent Categories
| Category | Description |
|----------|-------------|
| `Regulatory` | EFTA rules, compliance, identity verification, legal limits |
| `Consumer_Protection` | Consumer rights, fees, unauthorized charges, refunds |
| `Payment_Industry` | Payment processing, card networks, transfers, chargebacks |
| `Synthetic_Policies` | Internal procedures, account policies, dispute handling |

**API calls:** ~110 Groq calls (Cell 6 only) — all other cells are local

## Cell 0 — Repo Root & Config

In [ ]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

from config import BANKING77_TRAIN, BANKING77_TEST, REGULATORY_PATH

INTENT_DIR  = os.path.join('intent_classification')
os.makedirs(INTENT_DIR, exist_ok=True)

OUT_DATASET = os.path.join(INTENT_DIR, 'intent_dataset.csv')
REG_PDF     = os.path.join(REGULATORY_PATH, 'Regulation E.pdf')

print('Repo root   :', _root)
print('Banking77   :', BANKING77_TRAIN)
print('Reg PDF     :', REG_PDF)
print('Output CSV  :', OUT_DATASET)

## Cell 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q pdfplumber transformers openai python-dotenv matplotlib seaborn

## Cell 2 — Imports

In [ ]:
import re, json, time
import pdfplumber
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from transformers import AutoTokenizer
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY not found in .env')

GROQ_MODEL = 'meta-llama/llama-3.3-70b-instruct'
RATE_DELAY = 0.5

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Color palette for 4 categories — used consistently in all plots
COLORS = {
    'Regulatory':          '#E74C3C',  # red
    'Consumer_Protection': '#3498DB',  # blue
    'Payment_Industry':    '#2ECC71',  # green
    'Synthetic_Policies':  '#F39C12',  # orange
}

print('Imports OK')

## Cell 3 — Load Banking77 + Show Original Distribution

In [ ]:
train_df = pd.read_csv(BANKING77_TRAIN)
test_df  = pd.read_csv(BANKING77_TEST)
full_df  = pd.concat([train_df, test_df], ignore_index=True)

print('Banking77 loaded:')
print('  Train rows :', len(train_df))
print('  Test rows  :', len(test_df))
print('  Total rows :', len(full_df))
print('  Intents    :', full_df['category'].nunique())
print()
print('Sample queries:')
for _, row in full_df.sample(5, random_state=42).iterrows():
    print('  [{}]  {}'.format(row['category'], row['text'][:70]))

## Cell 4 — Map 77 Intents → 4 Categories + Drop Irrelevant
Mapping logic:
- **Consumer_Protection**: fees, unauthorized charges, refunds, lost cards
- **Payment_Industry**: card networks, transfers, payment processing, exchange
- **Synthetic_Policies**: account procedures, pending operations, top-up policies
- **Regulatory**: identity verification, compliance, age/country limits
- **DROP**: physical card management, app features, PIN, personal details

In [ ]:
# ── Intent mapping: Banking77 (77 classes) → your 4 PDF categories ────────────
# Only intents that have a meaningful PDF document match are kept.
# Intents about physical card delivery, PIN, app features = DROPPED (no PDF match)

INTENT_MAP = {
    # ── Consumer_Protection ───────────────────────────────────────────────────
    # Fees, unauthorized charges, refunds, disputed transactions
    'Refund_not_showing_up':                             'Consumer_Protection',
    'card_payment_fee_charged':                          'Consumer_Protection',
    'card_payment_not_recognised':                       'Consumer_Protection',
    'cash_withdrawal_charge':                            'Consumer_Protection',
    'cash_withdrawal_not_recognised':                    'Consumer_Protection',
    'compromised_card':                                  'Consumer_Protection',
    'direct_debit_payment_not_recognised':               'Consumer_Protection',
    'exchange_charge':                                   'Consumer_Protection',
    'extra_charge_on_statement':                         'Consumer_Protection',
    'lost_or_stolen_card':                               'Consumer_Protection',
    'request_refund':                                    'Consumer_Protection',
    'reverted_card_payment?':                            'Consumer_Protection',
    'transaction_charged_twice':                         'Consumer_Protection',
    'transfer_fee_charged':                              'Consumer_Protection',
    'wrong_amount_of_cash_received':                     'Consumer_Protection',
    'wrong_exchange_rate_for_cash_withdrawal':            'Consumer_Protection',
    'card_payment_wrong_exchange_rate':                  'Consumer_Protection',

    # ── Payment_Industry ──────────────────────────────────────────────────────
    # Card networks, payment processing, transfers, currency exchange
    'card_acceptance':                                   'Payment_Industry',
    'card_not_working':                                  'Payment_Industry',
    'contactless_not_working':                           'Payment_Industry',
    'exchange_rate':                                     'Payment_Industry',
    'exchange_via_app':                                  'Payment_Industry',
    'failed_transfer':                                   'Payment_Industry',
    'fiat_currency_support':                             'Payment_Industry',
    'pending_transfer':                                  'Payment_Industry',
    'receiving_money':                                   'Payment_Industry',
    'supported_cards_and_currencies':                    'Payment_Industry',
    'transfer_into_account':                             'Payment_Industry',
    'transfer_not_received_by_recipient':                'Payment_Industry',
    'transfer_timing':                                   'Payment_Industry',
    'visa_or_mastercard':                                'Payment_Industry',
    'declined_card_payment':                             'Payment_Industry',
    'declined_cash_withdrawal':                          'Payment_Industry',
    'declined_transfer':                                 'Payment_Industry',
    'card_about_to_expire':                              'Payment_Industry',

    # ── Synthetic_Policies ────────────────────────────────────────────────────
    # Internal bank procedures, account policies, pending operations
    'cancel_transfer':                                   'Synthetic_Policies',
    'beneficiary_not_allowed':                           'Synthetic_Policies',
    'balance_not_updated_after_bank_transfer':           'Synthetic_Policies',
    'balance_not_updated_after_cheque_or_cash_deposit':  'Synthetic_Policies',
    'pending_card_payment':                              'Synthetic_Policies',
    'pending_cash_withdrawal':                           'Synthetic_Policies',
    'pending_top_up':                                    'Synthetic_Policies',
    'top_up_by_bank_transfer_charge':                    'Synthetic_Policies',
    'top_up_by_card_charge':                             'Synthetic_Policies',
    'top_up_by_cash_or_cheque':                          'Synthetic_Policies',
    'top_up_failed':                                     'Synthetic_Policies',
    'top_up_limits':                                     'Synthetic_Policies',
    'top_up_reverted':                                   'Synthetic_Policies',
    'topping_up_by_card':                                'Synthetic_Policies',
    'verify_top_up':                                     'Synthetic_Policies',

    # ── Regulatory ────────────────────────────────────────────────────────────
    # Compliance rules, identity verification, legal requirements
    'verify_source_of_funds':                            'Regulatory',
    'unable_to_verify_identity':                         'Regulatory',
    'verify_my_identity':                                'Regulatory',
    'why_verify_identity':                               'Regulatory',
    'age_limit':                                         'Regulatory',
    'country_support':                                   'Regulatory',

    # ── DROP (no PDF document match) ──────────────────────────────────────────
    # activate_my_card, apple_pay_or_google_pay, atm_support, automatic_top_up
    # card_arrival, card_delivery_estimate, card_linking, card_swallowed
    # change_pin, disposable_card_limits, edit_personal_details
    # get_disposable_virtual_card, get_physical_card, getting_spare_card
    # getting_virtual_card, lost_or_stolen_phone, order_physical_card
    # passcode_forgotten, pin_blocked, terminate_account, virtual_card_not_working
}

# Apply mapping
full_df['intent'] = full_df['category'].map(INTENT_MAP)
mapped_df   = full_df[full_df['intent'].notna()][['text', 'intent']].reset_index(drop=True)
dropped_df  = full_df[full_df['intent'].isna()]

print('Banking77 mapping results:')
print('  Original rows  : {:,}'.format(len(full_df)))
print('  Kept rows      : {:,} ({:.1f}%)'.format(len(mapped_df), 100*len(mapped_df)/len(full_df)))
print('  Dropped rows   : {:,} ({:.1f}%)'.format(len(dropped_df), 100*len(dropped_df)/len(full_df)))
print()
print('Class distribution BEFORE augmentation:')
counts = mapped_df['intent'].value_counts()
for cat, n in counts.items():
    bar = '█' * (n // 50)
    print('  {:<25} {:>5,}  {}'.format(cat, n, bar))

## Cell 5 — Extract Regulation E Chunks (for Groq generation)
Extract Regulation E.pdf → token-exact chunks (200 tokens) → sample 110 chunks → generate 5 questions each = ~500 Regulatory rows.

In [ ]:
def extract_pdf_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text()
            if raw:
                clean = re.sub(r'\n{3,}', '\n\n', raw)
                clean = re.sub(r'[ \t]{2,}', ' ', clean)
                clean = re.sub(r'[^\x00-\x7F]+', ' ', clean).strip()
                if len(clean) > 50:
                    pages.append(clean)
    return '\n\n'.join(pages)


def chunk_token_exact(text, tokenizer, target=200, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    step   = target - overlap
    chunks = []
    start  = 0
    while start < len(tokens):
        end  = min(start + target, len(tokens))
        txt  = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if len(txt) > 30:
            chunks.append(txt)
        start += step
        if end == len(tokens):
            break
    return chunks


print('Extracting Regulation E.pdf...')
reg_text   = extract_pdf_text(REG_PDF)
reg_chunks = chunk_token_exact(reg_text, tokenizer, target=200, overlap=50)
print('  Total chunks from Regulation E: {:,}'.format(len(reg_chunks)))

# Sample 110 chunks — generating 5 questions each gives ~500-550 questions
# Extra 10 buffer in case some Groq calls fail
np.random.seed(42)
sampled_indices = np.random.choice(len(reg_chunks), size=min(110, len(reg_chunks)), replace=False)
sampled_chunks  = [reg_chunks[i] for i in sorted(sampled_indices)]
print('  Sampled for QA generation: {} chunks'.format(len(sampled_chunks)))
print('  Expected questions: ~{}'.format(len(sampled_chunks) * 5))

## Cell 6 — Generate 500 Regulatory Questions (Groq)
5 questions per chunk × 110 chunks = ~550 questions → keep 500.  
**API calls: ~110 Groq calls (~$0.02 total)**

In [ ]:
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
r = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': 'Reply: ready'}],
    max_tokens=5,
)
print('OpenRouter:', r.choices[0].message.content.strip())


def generate_regulatory_questions(chunk_text, client, n=5):
    """
    Generate n regulatory questions from a Regulation E chunk.
    Returns list of question strings.
    """
    prompt = (
        'You are a financial compliance expert. '
        'Read the regulatory text below and generate {} different questions '
        'that a compliance officer, banker, or consumer might ask about it. '
        'Questions must be answerable from this text. '
        'Output ONLY the questions, one per line, no numbering, no explanation.\n\n'
        'Regulatory Text:\n{}'
    ).format(n, chunk_text[:500])
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=200, temperature=0.4,
        )
        raw   = r.choices[0].message.content.strip()
        lines = [l.strip() for l in raw.split('\n') if len(l.strip()) > 10]
        return lines[:n]
    except Exception as e:
        print('  Error:', e)
        return []


print('Generating regulatory questions from {} chunks...'.format(len(sampled_chunks)))
regulatory_questions = []

for i, chunk in enumerate(tqdm(sampled_chunks)):
    questions = generate_regulatory_questions(chunk, client, n=5)
    regulatory_questions.extend(questions)
    time.sleep(RATE_DELAY)
    if (i + 1) % 20 == 0:
        print('  Progress: {}/{} chunks | {} questions so far'.format(
            i+1, len(sampled_chunks), len(regulatory_questions)))

# Keep exactly 500
regulatory_questions = list(set(regulatory_questions))  # deduplicate
regulatory_questions = regulatory_questions[:500]

reg_df = pd.DataFrame({
    'text':   regulatory_questions,
    'intent': 'Regulatory'
})

print('\nGenerated {} unique regulatory questions'.format(len(reg_df)))
print('Sample questions:')
for q in reg_df['text'].head(5):
    print('  -', q)

## Cell 7 — Combine All Data + Final Balanced Dataset

In [ ]:
# Combine Banking77 mapped + Groq-generated regulatory
final_df = pd.concat([mapped_df, reg_df], ignore_index=True)
final_df = final_df.drop_duplicates(subset='text').reset_index(drop=True)
final_df = final_df.rename(columns={'intent': 'category'})

print('Final dataset:')
print('  Total rows: {:,}'.format(len(final_df)))
print()
print('Class distribution AFTER augmentation:')
final_counts = final_df['category'].value_counts()
for cat, n in final_counts.items():
    bar = '█' * (n // 50)
    print('  {:<25} {:>5,}  {}'.format(cat, n, bar))

# Save
final_df.to_csv(OUT_DATASET, index=False)
print('\nSaved:', OUT_DATASET)

## Cell 8 — Visualization: Before vs After Augmentation
3 plots in one figure:
1. Banking77 original 77-class distribution (top 40 intents colored by category)
2. After mapping — 4-class distribution (shows Regulatory gap)
3. After augmentation — 4-class distribution (shows balance achieved)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Intent Dataset — Before vs After Augmentation', fontsize=14, fontweight='bold')

# ── Plot 1: Banking77 original — top 40 intents colored by mapped category ───
ax1 = axes[0]
orig_counts = full_df['category'].value_counts().head(40)

bar_colors = []
for intent in orig_counts.index:
    mapped = INTENT_MAP.get(intent)
    bar_colors.append(COLORS.get(mapped, '#95A5A6') if mapped else '#95A5A6')

bars = ax1.barh(range(len(orig_counts)), orig_counts.values, color=bar_colors)
ax1.set_yticks(range(len(orig_counts)))
ax1.set_yticklabels(orig_counts.index, fontsize=7)
ax1.set_xlabel('Query count')
ax1.set_title('Banking77 Original\n(Top 40 of 77 intents)', fontweight='bold')
ax1.invert_yaxis()

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in COLORS.items()]
legend_patches.append(mpatches.Patch(color='#95A5A6', label='DROPPED'))
ax1.legend(handles=legend_patches, fontsize=7, loc='lower right')

# ── Plot 2: After mapping — 4-class (before Groq augmentation) ───────────────
ax2 = axes[1]
before_counts = mapped_df['intent'].value_counts()
cats   = list(before_counts.index)
values = list(before_counts.values)
colors = [COLORS[c] for c in cats]

bars2 = ax2.bar(cats, values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             '{:,}'.format(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_title('After Mapping (before Groq)\nRegulatory severely underrepresented', fontweight='bold')
ax2.set_xlabel('Category')
ax2.set_ylabel('Query count')
ax2.set_xticklabels(cats, rotation=15, ha='right', fontsize=9)

# Add warning annotation on Regulatory bar
reg_idx = cats.index('Regulatory') if 'Regulatory' in cats else None
if reg_idx is not None:
    ax2.annotate('UNDERREPRESENTED',
                 xy=(reg_idx, before_counts['Regulatory']),
                 xytext=(reg_idx + 0.3, before_counts['Regulatory'] + 200),
                 fontsize=8, color='red', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='red'))

# ── Plot 3: After augmentation — 4-class (balanced) ──────────────────────────
ax3 = axes[2]
after_counts = final_df['category'].value_counts()
cats3   = list(after_counts.index)
values3 = list(after_counts.values)
colors3 = [COLORS[c] for c in cats3]

bars3 = ax3.bar(cats3, values3, color=colors3, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars3, values3):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             '{:,}'.format(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax3.set_title('After Groq Augmentation\n+500 Regulatory questions added', fontweight='bold')
ax3.set_xlabel('Category')
ax3.set_ylabel('Query count')
ax3.set_xticklabels(cats3, rotation=15, ha='right', fontsize=9)

# Add annotation showing what was added
if 'Regulatory' in cats3:
    reg_after  = after_counts['Regulatory']
    reg_before = before_counts.get('Regulatory', 0)
    ax3.annotate('+{} from Groq'.format(reg_after - reg_before),
                 xy=(cats3.index('Regulatory'), reg_after),
                 xytext=(cats3.index('Regulatory') + 0.3, reg_after + 200),
                 fontsize=8, color='green', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='green'))

plt.tight_layout()
plot_path = os.path.join('intent_classification', 'dataset_balance.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved:', plot_path)

# ── Summary stats ─────────────────────────────────────────────────────────────
print()
print('=' * 55)
print('  FINAL DATASET SUMMARY')
print('=' * 55)
print('  {:<25} {:>6} {:>8}'.format('Category', 'Count', 'Share%'))
print('-' * 55)
total = len(final_df)
for cat in ['Regulatory', 'Consumer_Protection', 'Payment_Industry', 'Synthetic_Policies']:
    n   = after_counts.get(cat, 0)
    pct = round(100 * n / total, 1)
    print('  {:<25} {:>6,} {:>7.1f}%'.format(cat, n, pct))
print('-' * 55)
print('  {:<25} {:>6,} {:>7.1f}%'.format('TOTAL', total, 100.0))
print('=' * 55)
print()
print('Dataset ready for: intent_classification/FinIntent_Classifier.ipynb')

---
## Cell 9 — Augment CSV with PDF-Domain Questions (3 remaining categories)
The existing CSV has 500 Groq Regulatory questions but 0 PDF-domain examples for the other 3 categories.
This causes domain mismatch at inference (Banking77-style ≠ real PDF-style queries).

**Fix:** generate ~200 questions per category from their source PDFs → append to existing CSV.
- Consumer_Protection  ← CFPB consumer toolkit PDF  
- Payment_Industry     ← Mastercard Chargeback Guide PDF  
- Synthetic_Policies   ← Chargeback filing / complaint procedures PDF  

**API calls: ~120 Groq calls (~$0.02 extra)**

In [ ]:
import glob as glob_mod

from config import CONSUMER_PROTECTION_PATH, PAYMENT_INDUSTRY_PATH

# ── Load existing CSV (preserves the 500 Regulatory questions already generated) ──
existing_df = pd.read_csv(OUT_DATASET)
print('Existing CSV loaded: {:,} rows'.format(len(existing_df)))
print('Current distribution:')
print(existing_df['category'].value_counts().to_string())
print()

# ── Find one PDF per category ─────────────────────────────────────────────────
def find_pdf(directory, keyword=None):
    """Return first PDF in directory, or the one matching keyword."""
    pdfs = sorted(glob_mod.glob(os.path.join(directory, '*.pdf')))
    if not pdfs:
        raise FileNotFoundError('No PDFs found in: {}'.format(directory))
    if keyword:
        matches = [p for p in pdfs if keyword.lower() in os.path.basename(p).lower()]
        if matches:
            return matches[0]
    return pdfs[0]

# Synthetic_Policies: check complaint_procedures first, then payment_industry folder
def find_synthetic_pdf():
    from config import COMPLAINT_PROCEDURES_PATH
    pdfs = sorted(glob_mod.glob(os.path.join(COMPLAINT_PROCEDURES_PATH, '*.pdf')))
    if pdfs:
        return pdfs[0]
    # fallback: chargeback policy PDF in payment industry folder
    pdfs = sorted(glob_mod.glob(os.path.join(PAYMENT_INDUSTRY_PATH, '*.pdf')))
    matches = [p for p in pdfs if 'policy' in p.lower() or 'filing' in p.lower() or 'chargeback' in p.lower()]
    return matches[0] if matches else pdfs[0]

PDF_MAP = {
    'Consumer_Protection': find_pdf(CONSUMER_PROTECTION_PATH),
    'Payment_Industry':    find_pdf(PAYMENT_INDUSTRY_PATH, keyword='mastercard'),
    'Synthetic_Policies':  find_synthetic_pdf(),
}

print('PDFs selected for augmentation:')
for cat, path in PDF_MAP.items():
    print('  [{:<22}]  {}'.format(cat, os.path.basename(path)))

## Cell 10 — Generate ~200 Questions per Category + Append to CSV
~40 chunks × 5 questions = ~200 per category × 3 categories = ~120 Groq calls.

In [ ]:
import re, time, os, glob as glob_mod
import numpy as np
import pandas as pd
import pdfplumber
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv
from transformers import AutoTokenizer

load_dotenv()
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
client     = OpenAI(api_key=OPENROUTER_API_KEY, base_url='https://openrouter.ai/api/v1')
GROQ_MODEL = 'meta-llama/llama-3.3-70b-instruct'
RATE_DELAY = 0.5

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# ── Helper functions (self-contained) ────────────────────────────────────────
def extract_pdf_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text()
            if raw:
                clean = re.sub(r'\n{3,}', '\n\n', raw)
                clean = re.sub(r'[ \t]{2,}', ' ', clean)
                clean = re.sub(r'[^\x00-\x7F]+', ' ', clean).strip()
                if len(clean) > 50:
                    pages.append(clean)
    return '\n\n'.join(pages)

def chunk_token_exact(text, tokenizer, target=200, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    step, chunks, start = target - overlap, [], 0
    while start < len(tokens):
        end = min(start + target, len(tokens))
        txt = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if len(txt) > 30:
            chunks.append(txt)
        start += step
        if end == len(tokens):
            break
    return chunks

CATEGORY_PROMPTS = {
    'Consumer_Protection': (
        'You are a consumer rights expert. Read the text below and generate {} different '
        'questions a consumer might ask about their rights, fees, refunds, unauthorized charges, '
        'or financial protection.\nOutput ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
    'Payment_Industry': (
        'You are a payments industry expert. Read the text below and generate {} different '
        'questions about payment processing, card networks, chargebacks, transfers, or '
        'declined transactions.\nOutput ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
    'Synthetic_Policies': (
        'You are a banking policy expert. Read the text below and generate {} different '
        'questions about internal bank procedures, dispute handling, account policies, '
        'complaint processes, or pending transaction rules.\nOutput ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
}

def generate_questions_for_category(pdf_path, category, client, tokenizer,
                                    n_chunks=40, n_per_chunk=5):
    print('  Extracting: {}'.format(os.path.basename(pdf_path)))
    text   = extract_pdf_text(pdf_path)
    chunks = chunk_token_exact(text, tokenizer, target=200, overlap=50)
    print('  Total chunks: {}  →  sampling {}'.format(len(chunks), n_chunks))

    np.random.seed(99)
    indices = np.random.choice(len(chunks), size=min(n_chunks, len(chunks)), replace=False)
    sampled = [chunks[i] for i in sorted(indices)]

    questions = []
    for chunk in tqdm(sampled, desc=category):
        prompt = CATEGORY_PROMPTS[category].format(n_per_chunk, chunk[:500])
        try:
            r = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=200, temperature=0.4,
            )
            raw   = r.choices[0].message.content.strip()
            lines = [l.strip() for l in raw.split('\n') if len(l.strip()) > 10]
            questions.extend(lines[:n_per_chunk])
        except Exception as e:
            print('  Error:', e)
        time.sleep(RATE_DELAY)

    questions = list(set(questions))[:200]
    print('  Generated {} unique questions for {}'.format(len(questions), category))
    return questions


# ── Load existing CSV + PDF paths ────────────────────────────────────────────
OUT_DATASET = os.path.join('intent_classification', 'intent_dataset.csv')
existing_df = pd.read_csv(OUT_DATASET)
print('Existing CSV: {:,} rows'.format(len(existing_df)))

from config import CONSUMER_PROTECTION_PATH, PAYMENT_INDUSTRY_PATH, COMPLAINT_PROCEDURES_PATH

def find_pdf(directory, keyword=None):
    pdfs = sorted(glob_mod.glob(os.path.join(directory, '*.pdf')))
    if not pdfs:
        raise FileNotFoundError('No PDFs in: {}'.format(directory))
    if keyword:
        matches = [p for p in pdfs if keyword.lower() in os.path.basename(p).lower()]
        if matches:
            return matches[0]
    return pdfs[0]

def find_synthetic_pdf():
    pdfs = sorted(glob_mod.glob(os.path.join(COMPLAINT_PROCEDURES_PATH, '*.pdf')))
    if pdfs:
        return pdfs[0]
    pdfs = sorted(glob_mod.glob(os.path.join(PAYMENT_INDUSTRY_PATH, '*.pdf')))
    matches = [p for p in pdfs if any(k in p.lower() for k in ['policy', 'filing', 'chargeback'])]
    return matches[0] if matches else pdfs[0]

PDF_MAP = {
    'Consumer_Protection': find_pdf(CONSUMER_PROTECTION_PATH),
    'Payment_Industry':    find_pdf(PAYMENT_INDUSTRY_PATH, keyword='mastercard'),
    'Synthetic_Policies':  find_synthetic_pdf(),
}
print('\nPDFs selected:')
for cat, path in PDF_MAP.items():
    print('  [{:<22}]  {}'.format(cat, os.path.basename(path)))


# ── Generate + append ────────────────────────────────────────────────────────
new_rows = []
for category, pdf_path in PDF_MAP.items():
    print('\n[{}]'.format(category))
    qs = generate_questions_for_category(pdf_path, category, client, tokenizer)
    for q in qs:
        new_rows.append({'text': q, 'category': category})

new_df       = pd.DataFrame(new_rows)
augmented_df = pd.concat([existing_df, new_df], ignore_index=True)
augmented_df = augmented_df.drop_duplicates(subset='text').reset_index(drop=True)
augmented_df.to_csv(OUT_DATASET, index=False)

print('\nUpdated CSV saved: {:,} total rows'.format(len(augmented_df)))
print(augmented_df['category'].value_counts().to_string())

## Cell 11 — Generate 600 More Questions per Category from All 25 PDFs
Uses existing `Dataset/chunks/chunks_512w.csv` (all 25 PDFs already chunked with category labels).
120 chunks × 5 questions = ~600 per category × 4 categories = ~480 Groq calls (~$0.04).
Appends to existing `intent_dataset.csv` — no re-generation of existing rows.

In [ ]:
import re, time, os
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
client     = OpenAI(api_key=OPENROUTER_API_KEY, base_url='https://openrouter.ai/api/v1')
GROQ_MODEL = 'meta-llama/llama-3.3-70b-instruct'
RATE_DELAY = 0.5

CHUNKS_CSV  = os.path.join('Dataset', 'chunks', 'chunks_512w.csv')
OUT_DATASET = os.path.join('intent_classification', 'intent_dataset.csv')

CATEGORIES = ['Regulatory', 'Consumer_Protection', 'Payment_Industry', 'Synthetic_Policies']

CATEGORY_PROMPTS = {
    'Regulatory': (
        'You are a financial compliance expert. Read the regulatory text below and generate '
        '{} specific questions a compliance officer or banker might ask about rules, '
        'legal requirements, identity verification, or government regulations.\n'
        'Output ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
    'Consumer_Protection': (
        'You are a consumer rights expert. Read the financial text below and generate '
        '{} specific questions a consumer might ask about their rights, fees, refunds, '
        'unauthorized charges, or financial protection.\n'
        'Output ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
    'Payment_Industry': (
        'You are a payments industry expert. Read the financial text below and generate '
        '{} specific questions about payment processing, card networks, chargebacks, '
        'fund transfers, or declined transactions.\n'
        'Output ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
    'Synthetic_Policies': (
        'You are a banking operations expert. Read the financial text below and generate '
        '{} specific questions about internal bank procedures, dispute handling, '
        'account policies, complaint processes, or pending transaction rules.\n'
        'Output ONLY the questions, one per line, no numbering.\n\nText:\n{}'
    ),
}

# ── Load all chunks + existing dataset ───────────────────────────────────────
chunks_df   = pd.read_csv(CHUNKS_CSV)
existing_df = pd.read_csv(OUT_DATASET)

print('Chunk corpus loaded: {:,} chunks across {} categories'.format(
    len(chunks_df), chunks_df['category'].nunique()))
print(chunks_df['category'].value_counts().to_string())
print()
print('Existing dataset: {:,} rows'.format(len(existing_df)))
print(existing_df['category'].value_counts().to_string())

# Track already-existing texts to avoid duplicates
existing_texts = set(existing_df['text'].str.strip().str.lower())


# ── Generate 600 questions per category ──────────────────────────────────────
N_CHUNKS_PER_CAT = 120   # 120 chunks × 5 questions = 600 per category
N_PER_CHUNK      = 5

new_rows = []

for category in CATEGORIES:
    cat_chunks = chunks_df[chunks_df['category'] == category]['text'].dropna().tolist()
    n_sample   = min(N_CHUNKS_PER_CAT, len(cat_chunks))
    print('\n[{}]  available chunks: {}  →  sampling {}'.format(
        category, len(cat_chunks), n_sample))

    np.random.seed(7)
    sampled = np.random.choice(cat_chunks, size=n_sample, replace=False).tolist()

    questions = []
    for chunk in tqdm(sampled, desc=category[:12]):
        prompt = CATEGORY_PROMPTS[category].format(N_PER_CHUNK, str(chunk)[:600])
        try:
            r = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=250, temperature=0.4,
            )
            raw   = r.choices[0].message.content.strip()
            lines = [l.strip() for l in raw.split('\n') if len(l.strip()) > 10]
            questions.extend(lines[:N_PER_CHUNK])
        except Exception as e:
            print('  Error:', e)
        time.sleep(RATE_DELAY)

    # Deduplicate against existing dataset
    added = 0
    for q in list(set(questions)):
        if q.strip().lower() not in existing_texts:
            new_rows.append({'text': q, 'category': category})
            existing_texts.add(q.strip().lower())
            added += 1

    print('  Added {} new questions for {}'.format(added, category))


# ── Append + save ─────────────────────────────────────────────────────────────
new_df       = pd.DataFrame(new_rows)
augmented_df = pd.concat([existing_df, new_df], ignore_index=True)
augmented_df = augmented_df.drop_duplicates(subset='text').reset_index(drop=True)
augmented_df.to_csv(OUT_DATASET, index=False)

print('\n' + '='*55)
print('  UPDATED DATASET')
print('='*55)
print('  Total rows: {:,}'.format(len(augmented_df)))
print()
for cat in CATEGORIES:
    n = len(augmented_df[augmented_df['category'] == cat])
    print('  {:<25} {:>6,}'.format(cat, n))
print('='*55)
print('\nSaved:', OUT_DATASET)